# Deploy Hosted Agents — `mafw-contoso` & `langgraph-contoso`

Hospeda os agentes locais (`src/app/mafwAgent`, `src/app/langgraphAgent`)
como **Hosted Agents** dentro do **Microsoft Foundry Agent Service**.
É o **mesmo código** dos containers locais — só muda o runtime e o entrypoint:

| Runtime | Dockerfile | Entrypoint | Porta | Protocolo |
|---|---|---|---|---|
| Local (`docker compose`) | `Dockerfile` | `uvicorn app:app` | 8091 / 8090 | `POST /chat` (custom) |
| Foundry Hosted | `Dockerfile.hosted` | `python -m responses_app` | 8088 | `POST /responses` (OpenAI Responses) |

O arquivo `responses_app.py` (em cada agente) embrulha o agente em
`ResponsesAgentServerHost` da lib `azure-ai-agentserver-responses` e reusa o
mesmo `ORCHESTRATOR` (MAFW) ou `GRAPH` (LangGraph) já implementados.

## Pré-requisitos

1. Projeto Foundry com **Hosted Agents (Preview)** disponível na região.
2. Você com **Azure AI Project Manager** no projeto.
3. ✅ ACR provisionado: `acrframeworkaicontoso.azurecr.io`.
4. ✅ Managed Identity do projeto Foundry com **AcrPull** no ACR.
5. ✅ Sua identidade com **AcrPush** no ACR.
6. `az login` + Docker Desktop rodando.
7. Variáveis em `infra/scripts/.env`:
   - `AZURE_AI_PROJECT_ENDPOINT`, `ACR_LOGIN_SERVER`, `HOSTED_TAG`
   - `AZURE_OPENAI_ENDPOINT`, `AZURE_OPENAI_CHAT_DEPLOYMENT`, `AZURE_SEARCH_ENDPOINT`

> **Nota:** Diferente do Azure ML, hosted agents do Foundry **não** exigem um
> objeto "Connection" para o ACR — basta o `AcrPull` na MI do projeto.

In [9]:
%pip install -q --upgrade "azure-ai-projects>=2.1.0" azure-identity python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [30]:
import os, time, subprocess, sys
from pathlib import Path
from dotenv import load_dotenv
load_dotenv(Path('.env'))

PROJECT_ENDPOINT = os.environ['AZURE_AI_PROJECT_ENDPOINT']
ACR              = os.environ['ACR_LOGIN_SERVER']           # ex: myacr.azurecr.io
MODEL_DEPLOY     = os.environ.get('AZURE_OPENAI_CHAT_DEPLOYMENT', 'gpt-4.1-mini')
AOAI_ENDPOINT    = os.environ.get('AZURE_OPENAI_ENDPOINT', '')
SEARCH_ENDPOINT  = os.environ.get('AZURE_SEARCH_ENDPOINT', '')
TAG = os.environ.get('HOSTED_TAG', 'v1')
REPO_ROOT = Path(__file__).resolve().parents[2] if '__file__' in globals() else Path.cwd().parents[1]
print('repo:', REPO_ROOT)
print('acr :', ACR)

repo: c:\Users\ptonpauletti\framework-ai-contoso
acr : acrframeworkaicontoso.azurecr.io


## 1) Build & push linux/amd64 das imagens

Usa o `Dockerfile.hosted` de cada agente — instala
`azure-ai-agentserver-responses` por cima do `requirements.txt` original e
roda `python -m responses_app` (porta 8088, protocolo Responses).

In [ ]:
AGENTS = [
    {
        'name': 'mafw-contoso',
        'context': str(REPO_ROOT),
        'dockerfile': 'src/app/mafwAgent/Dockerfile.hosted',
        'env': {
            'AZURE_OPENAI_ENDPOINT':           AOAI_ENDPOINT,
            'AZURE_OPENAI_CHAT_DEPLOYMENT':    MODEL_DEPLOY,
            'AZURE_SEARCH_ENDPOINT':           SEARCH_ENDPOINT,
            'AZURE_CLIENT_ID':                 '17bad333-f96a-4a41-9516-3d68d44f94a3',
        },
    },
    {
        'name': 'langgraph-contoso',
        'context': str(REPO_ROOT),
        'dockerfile': 'src/app/langgraphAgent/Dockerfile.hosted',
        'env': {
            'AZURE_OPENAI_ENDPOINT':           AOAI_ENDPOINT,
            'AZURE_OPENAI_CHAT_DEPLOYMENT':    MODEL_DEPLOY,
            'AZURE_SEARCH_ENDPOINT':           SEARCH_ENDPOINT,
            'AZURE_CLIENT_ID':                 '6de4ff9e-8470-471f-88de-23adba63edd2',
        },
    },
]

def sh(cmd):
    print('$', ' '.join(cmd))
    use_shell = sys.platform == 'win32'

    r = subprocess.run(' '.join(cmd) if use_shell else cmd,

                       shell=use_shell, capture_output=True, text=True)sh(['az', 'acr', 'login', '--name', ACR.split('.')[0]])

    if r.stdout:

        print(r.stdout)    return r.returncode

    if r.returncode != 0:        raise subprocess.CalledProcessError(r.returncode, cmd, r.stdout, r.stderr)
        print('STDERR:', r.stderr)

$ az acr login --name acrframeworkaicontoso
Login Succeeded



0

In [13]:
# Build + push de cada agente
for a in AGENTS:
    image = f"{ACR}/{a['name']}:{TAG}"
    dockerfile_abs = str(REPO_ROOT / a['dockerfile'])
    sh(['docker', 'build',
        '--platform', 'linux/amd64',
        '-f', dockerfile_abs,
        '-t', image,
        a['context']])
    sh(['docker', 'push', image])
    a['image'] = image
    print(' ->', image)

$ docker build --platform linux/amd64 -f c:\Users\ptonpauletti\framework-ai-contoso\src\app\mafwAgent\Dockerfile.hosted -t acrframeworkaicontoso.azurecr.io/mafw-contoso:v1 c:\Users\ptonpauletti\framework-ai-contoso
$ docker push acrframeworkaicontoso.azurecr.io/mafw-contoso:v1
The push refers to repository [acrframeworkaicontoso.azurecr.io/mafw-contoso]
57fb71246055: Waiting
01f59aef9b5c: Waiting
a13d59aff3fd: Waiting
2dc8d10c5e5f: Waiting
24bc7301b8f2: Waiting
2daaa613fa14: Waiting
6f92665ed17a: Waiting
fb4c70443787: Waiting
fd3f839387f5: Waiting
ba4290925138: Waiting
fd3f839387f5: Waiting
ba4290925138: Waiting
57fb71246055: Waiting
01f59aef9b5c: Waiting
a13d59aff3fd: Waiting
2dc8d10c5e5f: Waiting
24bc7301b8f2: Waiting
2daaa613fa14: Waiting
6f92665ed17a: Waiting
fb4c70443787: Waiting
2daaa613fa14: Waiting
6f92665ed17a: Waiting
fb4c70443787: Waiting
fd3f839387f5: Waiting
ba4290925138: Waiting
57fb71246055: Waiting
01f59aef9b5c: Waiting
a13d59aff3fd: Waiting
2dc8d10c5e5f: Waiting
24bc73

Exception in thread Thread-20 (_readerthread):
Traceback (most recent call last):
  File "C:\Python313\Lib\threading.py", line 1044, in _bootstrap_inner
    self.run()
    ~~~~~~~~^^
  File "C:\Python313\Lib\threading.py", line 995, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Python313\Lib\subprocess.py", line 1615, in _readerthread
    buffer.append(fh.read())
                  ~~~~~~~^^
  File "C:\Python313\Lib\encodings\cp1252.py", line 23, in decode
    return codecs.charmap_decode(input,self.errors,decoding_table)[0]
           ~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeDecodeError: 'charmap' codec can't decode byte 0x81 in position 1659: character maps to <undefined>


$ docker push acrframeworkaicontoso.azurecr.io/langgraph-contoso:v1
The push refers to repository [acrframeworkaicontoso.azurecr.io/langgraph-contoso]
1098114e1042: Waiting
c1123c2d0809: Waiting
01f59aef9b5c: Waiting
24bc7301b8f2: Waiting
18d58eee1c77: Waiting
fd3f839387f5: Waiting
6f92665ed17a: Waiting
57fb71246055: Waiting
fb4c70443787: Waiting
0706f8ca6ad3: Waiting
57fb71246055: Waiting
fb4c70443787: Waiting
0706f8ca6ad3: Waiting
1098114e1042: Waiting
c1123c2d0809: Waiting
01f59aef9b5c: Waiting
24bc7301b8f2: Waiting
18d58eee1c77: Waiting
fd3f839387f5: Waiting
6f92665ed17a: Waiting
fb4c70443787: Waiting
0706f8ca6ad3: Waiting
1098114e1042: Waiting
c1123c2d0809: Waiting
01f59aef9b5c: Waiting
24bc7301b8f2: Waiting
18d58eee1c77: Waiting
fd3f839387f5: Waiting
6f92665ed17a: Waiting
57fb71246055: Waiting
fb4c70443787: Waiting
0706f8ca6ad3: Waiting
1098114e1042: Waiting
c1123c2d0809: Waiting
01f59aef9b5c: Waiting
24bc7301b8f2: Waiting
18d58eee1c77: Waiting
fd3f839387f5: Waiting
6f92665ed17a:

## 2) Publica como Hosted Agent no Foundry

In [54]:
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import (
    HostedAgentDefinition, ProtocolVersionRecord, AgentProtocol,
)

project = AIProjectClient(
    endpoint=PROJECT_ENDPOINT,
    credential=DefaultAzureCredential(),
    allow_preview=True,
)

for a in AGENTS:
    a['image'] = f"{ACR}/{a['name']}:{TAG}"
    print(f"\n>>> create_version {a['name']} image={a['image']}")
    definition = HostedAgentDefinition(
        container_protocol_versions=[
            ProtocolVersionRecord(protocol=AgentProtocol.RESPONSES, version='1.0.0'),
        ],
        cpu='1',
        memory='2Gi',
        image=a['image'],
        environment_variables=a['env'],
    )
    version = project.agents.create_version(
        agent_name=a['name'],
        definition=definition,
    )
    a['version'] = version
    print('  created v=', version.version, ' name=', version.name)



>>> create_version mafw-contoso image=acrframeworkaicontoso.azurecr.io/mafw-contoso:debug
  created v= 8  name= mafw-contoso


In [53]:
TAG = 'v4'
print('TAG ->', TAG)


TAG -> debug


## 3) Aguarda ficar `active`

In [55]:
# A SDK 2.1.0 nao expoe campo de status; aguardamos um intervalo fixo
# para o controlador puxar a imagem e iniciar o container.
import time
for a in AGENTS:
    name, ver = a['name'], a['version'].version
    v = project.agents.get_version(agent_name=name, agent_version=ver)
    print(f'{name} v={ver} image={v.definition.image}')
print('\n... aguardando 60s para o agente subir ...')
time.sleep(60)
print('done')

mafw-contoso v=8 image=acrframeworkaicontoso.azurecr.io/mafw-contoso:debug

... aguardando 60s para o agente subir ...
done


## 4) Smoke-test via OpenAI Responses

Mesma chamada que o webapp faz na demo (`ato1_hosted_agents`).

In [60]:
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import OpenAI

token_provider = get_bearer_token_provider(
    DefaultAzureCredential(), 'https://ai.azure.com/.default'
)

for a in AGENTS:
    name = a['name']
    print(f'\n=== {name} ===')
    base_url = f"{PROJECT_ENDPOINT.rstrip('/')}/agents/{name}/endpoint/protocols/openai"
    client = OpenAI(
        api_key=token_provider,
        base_url=base_url,
        default_query={'api-version': 'v1'},
    )
    try:
        r = client.responses.create(input='ping', model=name)
        text = r.output_text if hasattr(r, 'output_text') else str(r.output)

        print('status:', r.status)

        print('OUT  :', text[:2000])        print('EXC', type(e).__name__, str(e)[:1500])
    except Exception as e:


=== mafw-contoso ===
URL: https://foundry-ai-framework.services.ai.azure.com/api/projects/proj-ai-framework/agents/mafw-contoso/endpoint/protocols/openai/responses?api-version=v1
HTTP 500
apim-request-id: 5bee00dd-8ab2-4972-b9a4-8dc801481717
BODY: {"error":{"message":"internal server error","type":"server_error","code":"server_error","param":null}}


In [57]:
for a in AGENTS:
    name, ver = a['name'], a['version'].version
    v = project.agents.get_version(agent_name=name, agent_version=ver)
    print(f"\n=== {name} v={ver} ===")
    print('attrs:', [x for x in dir(v) if not x.startswith('_')])
    print('as_dict:', v.as_dict() if hasattr(v, 'as_dict') else v.__dict__)



=== mafw-contoso v=8 ===
attrs: ['agent_guid', 'as_dict', 'blueprint', 'blueprint_reference', 'clear', 'copy', 'created_at', 'definition', 'description', 'get', 'id', 'instance_identity', 'items', 'keys', 'metadata', 'name', 'object', 'pop', 'popitem', 'setdefault', 'status', 'update', 'values', 'version']
as_dict: {'metadata': {}, 'object': 'agent.version', 'id': 'mafw-contoso:8', 'name': 'mafw-contoso', 'version': '8', 'description': '', 'created_at': 1778627395, 'definition': {'kind': 'hosted', 'container_protocol_versions': [{'protocol': 'responses', 'version': '1.0.0'}], 'cpu': '1', 'memory': '2Gi', 'environment_variables': {'AZURE_OPENAI_ENDPOINT': 'https://foundry-ai-framework.openai.azure.com/', 'AZURE_OPENAI_CHAT_DEPLOYMENT': 'gpt-4.1-mini', 'AZURE_SEARCH_ENDPOINT': 'https://ai-search-ai-framework.search.windows.net', 'AZURE_CLIENT_ID': '17bad333-f96a-4a41-9516-3d68d44f94a3'}, 'image': 'acrframeworkaicontoso.azurecr.io/mafw-contoso:debug'}, 'status': 'active', 'instance_ident

## 5) Variáveis para o webapp

Adicione no `.env` do webapp (ou do compose):

```env
AZURE_AI_HOSTED_MAFW=mafw-contoso
AZURE_AI_HOSTED_LANGGRAPH=langgraph-contoso
```

Então abra `http://localhost:8080/sections/ato1_hosted_agents` e
alterne entre **Local container** e **Foundry Hosted** — mesmo código,
dois runtimes.